In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip uninstall -y datasets

Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1


In [2]:
!pip install datasets==2.17

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 9.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
ERROR: pip's dependency resolv

In [3]:
import json
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer
from transformers import RobertaForTokenClassification
from sklearn.metrics import accuracy_score, classification_report
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

2025-12-08 11:26:55.762851: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765193215.960622      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765193216.015805      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
# Load dataset
dataset_restaurant = load_dataset("jakartaresearch/semeval-absa", name='restaurant')
train_ds = dataset_restaurant["train"]
test_ds = dataset_restaurant["validation"]

Generating train split:   0%|          | 0/3044 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
# Label map
label_map = {"O": 0, "B": 1, "I": 2}
id2label = {v: k for k, v in label_map.items()}

In [7]:
#load bert model
model = RobertaForTokenClassification.from_pretrained("roberta-base", num_labels=len(label_map))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
#Load bert tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base",
    use_fast=True   # Force Fast tokenizer
)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [9]:
# Apply PEFT with LoRA
lora_config = LoraConfig(
    r=8,  # Rank of the LoRA matrix
    lora_alpha=16,  # Scaling factor
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.TOKEN_CLS  # Assuming token classification task
)
peft_model = get_peft_model(model, lora_config)
peft_model.to(device)
peft_model.train()

PeftModelForTokenClassification(
  (base_model): LoraModel(
    (model): RobertaForTokenClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(50265, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDic

In [10]:
# Tokenize
def tokenize_and_align(ex):
    text = ex["text"]
    terms = ex["aspects"]["term"]
    from_idx = ex["aspects"]["from"]
    to_idx = ex["aspects"]["to"]

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    labels = ["O"] * len(encoding["offset_mapping"][0])

    for start, end in zip(from_idx, to_idx):
        span_token_indices = []

        # collect all token positions belonging to this aspect span
        for i, (s, e) in enumerate(encoding["offset_mapping"][0]):
            if s == 0 and e == 0:   # padding tokens
                continue
            if s >= start and e <= end:
                span_token_indices.append(i)

        # assign BIO tags
        if len(span_token_indices) > 0:
            labels[span_token_indices[0]] = "B"       # first token
            for idx in span_token_indices[1:]:        # remaining tokens
                labels[idx] = "I"

    label_ids = [label_map[l] for l in labels]

    return (
        encoding["input_ids"].squeeze(),
        encoding["attention_mask"].squeeze(),
        torch.tensor(label_ids)
    )


In [11]:
# Build tensors
def prepare(ds):
    input_ids, attention_masks, label_ids = [], [], []
    for ex in ds:
        a, b, c = tokenize_and_align(ex)
        input_ids.append(a)
        attention_masks.append(b)
        label_ids.append(c)
    return TensorDataset(torch.stack(input_ids), torch.stack(attention_masks), torch.stack(label_ids))



In [12]:
train_dataset = prepare(train_ds)
test_dataset = prepare(test_ds)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=5e-5)

In [17]:
# Training loop
for epoch in range(3):
    for batch in train_loader:
        input_ids, attention_mask, labels = [x.to(device) for x in batch]
        outputs = peft_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch + 1} loss: {loss.item()}")

Epoch 1 loss: 0.004286648705601692
Epoch 2 loss: 0.0002568966301623732
Epoch 3 loss: 0.006048842798918486


In [51]:
def bio_to_spans(tokens, labels):
    spans = []
    current = []

    for tok, lab in zip(tokens, labels):
        tok = tok.replace("Ġ", " ").strip()  # remove subword marker reliably
        if lab == "B":
            if current:
                spans.append(" ".join(current))
            current = [tok]
        elif lab == "I" and current:
            current.append(tok)
        else:
            if current:
                spans.append(" ".join(current))
                current = []
    if current:
        spans.append(" ".join(current))
    return [s.strip() for s in spans]


In [58]:
!pip install rapidfuzz

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.0 MB/s eta 0:00:00a 0:00:01


In [59]:
import re
import string

from rapidfuzz import fuzz  # pip install rapidfuzz

def normalize_span(x):
    """Clean and normalize predicted or gold span text"""
    x = re.sub(r"(^Ġ+|\s+Ġ+)", " ", x)
    x = x.lower().strip()
    x = x.translate(str.maketrans("", "", string.punctuation.replace("-", "").replace("/", "")))
    x = re.sub(r"\s+", " ", x)
    edge_stop = ["the", "a", "an", "and", "or", "for", "of", "to", "with", "is", "was", "were"]
    tokens = x.split()
    while tokens and tokens[0] in edge_stop:
        tokens.pop(0)
    while tokens and tokens[-1] in edge_stop:
        tokens.pop()
    return " ".join(tokens).strip()



In [60]:
def remove_subspans(spans):
    """Keep only the longest spans in case of overlapping / sub-spans"""
    spans = sorted(spans, key=len, reverse=True)
    filtered = []
    for s in spans:
        if not any(s in f for f in filtered):
            filtered.append(s)
    return filtered


In [61]:
def calculate_span_metrics(gold_data, pred_data):
    """
    Compute span-level Precision, Recall, F1, Exact-Match Accuracy,
    and Overall Accuracy (average fraction of correct spans per sentence).
    """
    tp, fp, fn = 0, 0, 0
    exact_matches = 0
    per_sample_accuracies = []

    for gold_list, pred_list in zip(gold_data, pred_data):
        gold_set = set([str(g).lower().strip() for g in gold_list])
        pred_set = set([str(p).lower().strip() for p in pred_list])

        # Count correct spans
        correct = len(gold_set & pred_set)
        total = len(gold_set)

        # Per-sample accuracy (fraction of gold spans correctly predicted)
        if total > 0:
            per_sample_accuracies.append(correct / total)
        else:
            per_sample_accuracies.append(0)

        # For span-level metrics
        tp += correct
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

        # For exact-match accuracy
        if gold_set == pred_set:
            exact_matches += 1

    # Span-level Precision, Recall, F1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # Exact-match accuracy
    accuracy_em = exact_matches / len(gold_data) if len(gold_data) > 0 else 0

    # Overall accuracy (average fraction of correct spans per sentence)
    accuracy_overall = sum(per_sample_accuracies) / len(per_sample_accuracies)

    return {
        "Precision": round(precision * 100, 2),
        "Recall": round(recall * 100, 2),
        "F1 Score": round(f1 * 100, 2),
        "Accuracy (Overall)": round(accuracy_overall * 100, 2),
        "Accuracy (Exact Match)": round(accuracy_em * 100, 2)
    }


In [62]:
pred_spans_list = []
gold_spans_list = []

peft_model.eval()

with torch.no_grad():
    for ex in test_ds:
        text = ex["text"]
        gold_spans = ex["aspects"]["term"]

        encoding = tokenizer(text, return_tensors="pt", truncation=True).to(device)
        logits = peft_model(**encoding).logits
        preds = torch.argmax(logits, dim=2)[0].cpu().numpy()

        tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
        labels = [id2label[p] for p in preds]

        # Convert BIO → aspect spans
        pred_spans = bio_to_spans(tokens, labels)

        pred_spans_list.append(pred_spans)
        gold_spans_list.append(gold_spans)


In [63]:
metrics = calculate_span_metrics(gold_spans_list, pred_spans_list)

print("Span-level Metrics:")
print(f"Precision: {metrics['Precision']}%")
print(f"Recall: {metrics['Recall']}%")
print(f"F1 Score: {metrics['F1 Score']}%")
print(f"Overall Accuracy: {metrics['Accuracy (Overall)']}%")
print(f"Exact-Match Accuracy: {metrics['Accuracy (Exact Match)']}%")


Span-level Metrics:
Precision: 51.47%
Recall: 55.36%
F1 Score: 53.35%
Overall Accuracy: 51.29%
Exact-Match Accuracy: 32.12%
